# Requirements Extraction: PDF → Qwen VL → JSON → Neo4j

This notebook runs the full pipeline:
1. **Load a PDF** and stitch two pages into one image
2. **Run the Qwen VL model** to extract structured entries and classify requirements
3. **Parse the JSON** and filter to requirements only
4. **Store requirements in Neo4j** and explore the graph

You'll also get a short **Neo4j primer** and example Cypher queries to understand the graph.

## 1. Setup

Add project root to path and load config. Set `PDF_PATH` and optionally `FIRST_PAGE` (0-based index of the first of the two pages to stitch).

In [3]:
from src.neo4j_loader import get_driver

driver = get_driver()
with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print("Neo4j connection OK:", result.single()["test"])
driver.close()

Neo4j connection OK: 1


In [ ]:
import sys
from pathlib import Path

# Project root (folder containing run_pipeline.py)
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# --- CONFIG: set your PDF and page range ---
PDF_PATH = ROOT / "path" / "to" / "your" / "document.pdf"  # change this
FIRST_PAGE = 0  # 0 = stitch pages 1 & 2; 2 = stitch pages 3 & 4

## 2. Stitch two PDF pages into one image

The model expects a single image. We render two consecutive pages and stack them vertically.

In [ ]:
from src.pdf_utils import stitch_pdf_pages

image, page_numbers = stitch_pdf_pages(PDF_PATH, first_page=FIRST_PAGE)
print(f"Stitched image size: {image.size}, pages (1-based): {page_numbers}")

try:
    from IPython.display import display
    display(image)
except Exception:
    pass

## 3. Load model and run inference

Load the Qwen VL model/processor (from `QWEN_MODEL_PATH` in env or config) and generate the extraction JSON from the stitched image.

In [ ]:
from config import DEVICE
from src.inference import load_model_and_processor, run_inference

print("Loading model and processor (may take a moment)...")
model, processor, torch_dtype = load_model_and_processor()
print(f"Device: {DEVICE}, dtype: {torch_dtype}")

In [ ]:
print("Running inference...")
raw_output = run_inference(image, model, processor, DEVICE, torch_dtype)
print("Raw model output:")
print(raw_output[:2000])
if len(raw_output) > 2000:
    print("... [truncated]")

## 4. Parse JSON and get requirements

Extract the JSON from the model output (handles markdown code blocks or raw `{...}`) and validate against our schema. Then filter to entries where `is_requirement` is "yes"/"true"/"1".

In [ ]:
from src.json_utils import parse_extraction

result = parse_extraction(raw_output)
# Ensure meta.pages for the two-page image
if "pages" not in result.meta:
    result.meta["pages"] = page_numbers

requirements = result.requirements_only()
print(f"Total entries: {len(result.entries)}")
print(f"Requirements (is_requirement=yes): {len(requirements)}")
print("\n--- Sample requirement ---")
if requirements:
    r = requirements[0]
    print(f"Section: {r.section_title}\nPage: {r.page_number}\nText: {r.text[:200]}...")

## 4b. Assign tiers with Nova Pro (plant / system / component)

After filtering to requirements, we call **Amazon Nova Pro** (Bedrock) to classify each requirement as **plant**, **system**, or **component**. Set `AWS_REGION` and ensure AWS credentials are configured (env or `~/.aws/credentials`). Optional: set `NOVA_MODEL_ID` in `.env` (default `us.amazon.nova-pro-v1:0`).

In [ ]:
from src.nova_tier import assign_tiers_to_requirements

requirements_with_tiers = None
if requirements:
    try:
        requirements_with_tiers = assign_tiers_to_requirements(requirements)
        for e, tier in requirements_with_tiers:
            print(f"  [{tier}] {e.text[:70]}...")
    except Exception as ex:
        print("Nova Pro tier assignment failed:", ex)
        print("You can still load without tiers (legacy loader) in the next step.")

## 5. Understanding Neo4j

**Neo4j** is a **graph database**: instead of tables and rows, it stores **nodes** (entities) and **relationships** (connections between them). That makes it a good fit for requirements: you can link documents → sections → requirements, and later add links between requirements (e.g. "depends on", "refines").

### Core ideas

| Concept | Meaning | Example |
|--------|---------|--------|
| **Node** | One entity, with a **label** (type) and **properties** (key-value) | `(Document {id: "SRS", title: "Software Requirements"})` |
| **Relationship** | A directed link between two nodes, with a **type** and optional properties | `(Document)-[:HAS_SECTION]->(Section)` |
| **Cypher** | The query language (like SQL for graphs) | `MATCH (d:Document) RETURN d.title` |

### Cypher in 30 seconds

- **MATCH** – describe the pattern you’re looking for (nodes and relationships).
- **RETURN** – what to give back (nodes, properties, counts).
- **CREATE / MERGE** – create nodes/relationships (MERGE = “create if not exists”).
- **WHERE** – filter on properties.
- **Parameters** – use `$param` in the query and pass `{"param": value}` for safety.

Example: *"Find all requirements in section '3.1'"*

```cypher
MATCH (s:Section {title: "3.1"})-[:CONTAINS]->(r:Requirement)
RETURN r.text, r.page_number
```

### Our graph shape

We build this structure:

```
(Document) -[:HAS_SECTION]-> (Section) -[:CONTAINS]-> (Requirement)
```

- **Document**: `id`, `title`, `pages` (list of page numbers).
- **Section**: `doc_id`, `title`, `doc_title`; one per (document, section).
- **Requirement**: `id`, `text`, `page_number`, `section_title`, `doc_title`.

So: one document has many sections; each section contains many requirements. You can later add relationships like `(Requirement)-[:REFERENCES]->(Requirement)` if you parse cross-references.

## 6. Connect to Neo4j and load requirements

Ensure Neo4j is running (e.g. Docker: `docker run -p 7687:7687 -e NEO4J_AUTH=neo4j/yourpassword neo4j`) and set `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` in your environment or `.env`.

In [ ]:
from src.neo4j_loader import get_driver, load_requirements_into_neo4j, load_tiered_requirements_into_neo4j

driver = get_driver()
if requirements_with_tiers:
    n = load_tiered_requirements_into_neo4j(requirements_with_tiers, driver=driver)
    print(f"Loaded {n} tiered requirement nodes (Plant/System/Component) into Neo4j.")
else:
    n = load_requirements_into_neo4j(result, driver=driver)
    print(f"Loaded {n} requirement nodes into Neo4j (legacy).")
driver.close()

## 7. Explore the graph with Cypher

Run these queries to see how the graph looks. Change them to match your data (e.g. document id or section title). You can also open **Neo4j Browser** at `http://localhost:7474` and run the same Cypher there to see the graph visually.

In [ ]:
driver = get_driver()

def run_query(query: str, params: dict = None):
    """Run a Cypher query and print results as a simple table."""
    with driver.session() as session:
        res = session.run(query, params or {})
        records = list(res)
    if not records:
        print("(no results)")
        return
    keys = list(records[0].keys())
    print("\t".join(keys))
    print("-" * 60)
    for r in records:
        print("\t".join(str(r[k])[:50] for k in keys))
    return records

In [ ]:
# List all documents
print("=== All documents ===")
run_query("MATCH (d:Document) RETURN d.id AS id, d.title AS title, d.pages AS pages")

In [ ]:
# For each document: list its sections
print("=== Document → Sections ===")
run_query("""
    MATCH (d:Document)-[:HAS_SECTION]->(s:Section)
    RETURN d.title AS doc, s.title AS section
    ORDER BY d.title, s.title
""")

In [ ]:
# Count requirements per section
print("=== Requirement count per section ===")
run_query("""
    MATCH (s:Section)-[:CONTAINS]->(r:Requirement)
    RETURN s.doc_title AS doc, s.title AS section, count(r) AS req_count
    ORDER BY req_count DESC
""")

In [ ]:
# Get full requirement text for one section (replace "3.1" with your section title if needed)
print("=== Sample requirements (first section) ===")
run_query("""
    MATCH (s:Section)-[:CONTAINS]->(r:Requirement)
    WITH s, r LIMIT 5
    RETURN r.section_title AS section, r.page_number AS page, r.text AS text
""")

In [ ]:
driver.close()
print("\nDone. Use Neo4j Browser (http://localhost:7474) to visualize the graph interactively.")

### Cypher cheat sheet

| Goal | Cypher |
|------|--------|
| All nodes of a label | `MATCH (n:Document) RETURN n` |
| Filter by property | `MATCH (d:Document {id: "MyDoc"}) RETURN d` |
| Follow a relationship | `MATCH (d:Document)-[:HAS_SECTION]->(s) RETURN s` |
| Variable-length path | `MATCH (d:Document)-[:HAS_SECTION*]->(r:Requirement) RETURN r` |
| Count | `MATCH (r:Requirement) RETURN count(r)` |
| Parameterized (safe) | `MATCH (d:Document {id: $doc_id}) RETURN d` then pass `{"doc_id": "..."}` |

In Neo4j Browser you can also run `:schema` to see node labels and relationship types, or click a node and click "Expand" to traverse the graph.